In [3]:
from pypdf import PdfReader

# Load one PDF to start
reader = PdfReader("../data/property_policy.pdf")

# See how many pages it has
print(f"Number of pages: {len(reader.pages)}")

# Read the first page
first_page = reader.pages[4]
print("\nFirst page text:")
print(first_page.extract_text())

Number of pages: 36

First page text:
 Property Insurance Policy | P2
General conditions  
applicable to Property Damage 
insurance Business interruption 
insurance Money insurance and 
T errorism insurance
1  Policy Voidable
  This Policy shall be voidable in the event of misrepresentation 
misdescription or non-disclosure in any material particular
2  Observance of T erms
  It is a requirement of this Policy that liability of the Company is 
conditional upon observance of the terms of this Policy relating 
to anything to be done or complied with by the Policyholder  This 
shall include any requirements described in this Policy or any clause 
attaching to and forming part of this Policy as condition precedents 
to any liability of the Company
3  Reasonable Precautions
 The Policyholder at his own expense shall
 (A)  take all reasonable precautions to prevent or diminish loss 
destruction or damage or any occurrence or cease any activity 
which may give rise to liability under this Pol

In [4]:
# Extract all text from the PDF
all_text = ""

for page in reader.pages:
    all_text += page.extract_text()

# See how much text we got
print(f"Total characters extracted: {len(all_text)}")
print("\nFirst 500 characters:")
print(all_text[:500])

Total characters extracted: 101860

First 500 characters:
PROPERTY insuRancE
PolicyTHIS POLICY (AND THE SCHEDULE WHICH FORMS AN INTEGRAL PART OF THE POLICY) IS A LEGAL CONTRACT .
IT NEEDS TO BE EXAMINED THOROUGHLY TO ENSURE IT MEETS THE INSURED’S REQUIREMENTS. IF IT DOES NOT
MEET THE INSURED’S REQUIREMENTS THE INSURANCE ADVISER NEEDS TO BE CONTACTED WITHOUT UNDUE
DELAY
ANY FACTS  WHICH  THE INSURER HAS TAKEN INTO ACCOUNT IN THE ASSESSMENT OR  ACCEPTANCE OF THIS
INSURANCE, AND ANY SUBSEQUENT CHANGES TO THOSE FACTS, NEED TO BE DECLARED. FAILURE TO DO SO



In [5]:
import os

# Path to our data folder
data_folder = "../data"

# Dictionary to store text from each PDF
documents = {}

# Loop through every PDF in the data folder
for filename in os.listdir(data_folder):
    if filename.endswith(".pdf"):
        filepath = os.path.join(data_folder, filename)
        reader = PdfReader(filepath)
        
        # Extract all pages
        text = ""
        for page in reader.pages:
            text += page.extract_text()
        
        documents[filename] = text
        print(f"{filename}: {len(text)} characters extracted")

print(f"\nTotal documents loaded: {len(documents)}")

commercial_combined_insurance_policy.pdf: 332617 characters extracted
freight_solutions_policy.pdf: 60192 characters extracted


Ignoring wrong pointing object 20 0 (offset 0)
Ignoring wrong pointing object 22 0 (offset 0)
Ignoring wrong pointing object 55 0 (offset 0)


legacy_policy.pdf: 215696 characters extracted
office_policy.pdf: 248582 characters extracted
property_owners_policy.pdf: 108299 characters extracted
property_policy.pdf: 101860 characters extracted

Total documents loaded: 6


**Note** — each document gets chunked individually, not each page, and not all 6 PDFs as one giant blob.
So the flow looks like this:

commercial_combined_insurance_policy.pdf → 600+ chunks
freight_solutions_policy.pdf → 120+ chunks
legacy_policy.pdf → 400+ chunks
office_policy.pdf → 500+ chunks
property_owners_policy.pdf → 200+ chunks
property_policy.pdf → 200+ chunks

In [6]:
def chunk_text(text, chunk_size=500, overlap=50):
    chunks = []
    start = 0
    
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start = end - overlap
    
    return chunks

# Test on one document first
test_chunks = chunk_text(documents["property_policy.pdf"])

print(f"Total chunks created: {len(test_chunks)}")
print(f"\nFirst chunk:")
print(test_chunks[0])
print(f"\nSecond chunk:")
print(test_chunks[1])

Total chunks created: 227

First chunk:
PROPERTY insuRancE
PolicyTHIS POLICY (AND THE SCHEDULE WHICH FORMS AN INTEGRAL PART OF THE POLICY) IS A LEGAL CONTRACT .
IT NEEDS TO BE EXAMINED THOROUGHLY TO ENSURE IT MEETS THE INSURED’S REQUIREMENTS. IF IT DOES NOT
MEET THE INSURED’S REQUIREMENTS THE INSURANCE ADVISER NEEDS TO BE CONTACTED WITHOUT UNDUE
DELAY
ANY FACTS  WHICH  THE INSURER HAS TAKEN INTO ACCOUNT IN THE ASSESSMENT OR  ACCEPTANCE OF THIS
INSURANCE, AND ANY SUBSEQUENT CHANGES TO THOSE FACTS, NEED TO BE DECLARED. FAILURE TO DO SO


Second chunk:
HOSE FACTS, NEED TO BE DECLARED. FAILURE TO DO SO
MAY INVALIDATE  YOUR POLICY OR RESULT IN CERTAIN COVERS NOT OPERATING FULLY IF YOU ARE IN ANY
DOUBT AS TO WHETHER A FACT IS MATERIAL OR NOT ,  THE INSURANCE ADVISER NEEDS TO BE CONTACTED
WITHOUT UNDUE DELAY
RSA Insurance Ireland DAC (herein called the Company) and the Policyholder agree that
This Policy the Schedule (including any Schedule issued in substitution) and any Memoranda shall be c

Then all those chunks from all 6 documents get stored together in ChromaDB — our vector database. Each chunk carries a label telling us which document it came from and roughly where.

So when a question comes in, ChromaDB searches across ALL chunks from ALL documents at once and finds the most relevant ones — regardless of which PDF they came from.

**Why not chunk page by page?**
Pages are an arbitrary boundary — a sentence or clause can start on page 4 and finish on page 5. If we chunk by page, we'd break that sentence in half and lose its meaning. Chunking by character count with overlap is cleaner and more reliable.

**Why not one giant blob of all 6 PDFs together?**
Because we'd lose track of which document an answer came from. We need to be able to say "Source: property_policy.pdf, chunk 42" — that's how citations work.


In [7]:
# Chunk all documents
all_chunks = []

for filename, text in documents.items():
    chunks = chunk_text(text)
    
    for i, chunk in enumerate(chunks):
        all_chunks.append({
            "text": chunk,
            "source": filename,
            "chunk_id": i
        })

print(f"Total chunks across all documents: {len(all_chunks)}")
print(f"\nExample chunk:")
print(f"Source: {all_chunks[0]['source']}")
print(f"Chunk ID: {all_chunks[0]['chunk_id']}")
print(f"Text: {all_chunks[0]['text']}")

Total chunks across all documents: 2375

Example chunk:
Source: commercial_combined_insurance_policy.pdf
Chunk ID: 0
Text: FLAT SIZE:  297MM H   420MM W
FINISHED SIZE: 297MM H   210MM W
Commercial  
Combined Insurance
Policy Wording1 | Commercial Combined Insurance Policy Wording
Contents
Section Page Number
Introduction.......................................................................................................................................................................2
About Your Policy...................................................................................................................


In [8]:
from sentence_transformers import SentenceTransformer

# Load the embedding model
# This will download the model the first time (~90MB) - just wait for it
model = SentenceTransformer('all-MiniLM-L6-v2')

# Test it on one chunk first
test_chunk = all_chunks[0]['text']
embedding = model.encode(test_chunk)

print(f"Chunk text: {test_chunk[:100]}...")
print(f"\nEmbedding shape: {embedding.shape}")
print(f"First 5 numbers: {embedding[:5]}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Chunk text: FLAT SIZE:  297MM H   420MM W
FINISHED SIZE: 297MM H   210MM W
Commercial  
Combined Insurance
Polic...

Embedding shape: (384,)
First 5 numbers: [ 0.0560066   0.02702003 -0.05864294 -0.05581209  0.00853023]


***The model all-MiniLM-L6-v2 was designed and trained to output 384 dimensions. That's a fixed property of the model itself, not something we control.***

<u>**Now let's embed all 2,375 chunks.**</u>

In [9]:
# Extract just the text from all 2375 chunks
texts = [chunk['text'] for chunk in all_chunks]

# Embed all chunks at once
# batch_size=32 means it processes 32 chunks at a time
print("Embedding all chunks... this will take a few minutes")
embeddings = model.encode(texts, batch_size=32, show_progress_bar=True)

print(f"\nDone!")
print(f"Total embeddings created: {embeddings.shape[0]}")
print(f"Each embedding has {embeddings.shape[1]} numbers")

Embedding all chunks... this will take a few minutes


Batches:   0%|          | 0/75 [00:00<?, ?it/s]


Done!
Total embeddings created: 2375
Each embedding has 384 numbers


In [10]:
import chromadb

# Create a ChromaDB client that saves to disk
client = chromadb.PersistentClient(path="../data/chromadb")

# Create a collection - think of this like a table in a database
collection = client.get_or_create_collection(
    name="insurance_policies",
    metadata={"hnsw:space": "cosine"}
)

print("ChromaDB collection created successfully!")
print(f"Collection name: {collection.name}")

ChromaDB collection created successfully!
Collection name: insurance_policies


In [11]:
# How many items are in the collection?
print(f"Number of items in collection: {collection.count()}")

Number of items in collection: 2375


**ChromaDB stores 4 things for each chunk:**


<i>**ID**          → a unique label for each chunk
             e.g. "commercial_combined_0"

<i>**Document**    → the actual text of the chunk
             e.g. "Water damage is covered when..."

<i>**Embedding**   → the 384 numbers representing meaning
             e.g. [0.056, 0.027, -0.058, ...]

<i>**Metadata**    → extra info about the chunk
             e.g. {"source": "property_policy.pdf", "chunk_id": 42}

In [12]:
# Check collection details
print(f"Collection name: {collection.name}")
print(f"Collection ID: {collection.id}")
print(f"Collection metadata: {collection.metadata}")

Collection name: insurance_policies
Collection ID: 80bdb032-9b63-4eb6-b42c-08890e1d773c
Collection metadata: {'hnsw:space': 'cosine'}


Now let's add all 2,375 chunks into ChromaDB. 

In [13]:
# Prepare the 4 things ChromaDB needs
ids = [f"{chunk['source']}_{chunk['chunk_id']}" for chunk in all_chunks]
texts_list = [chunk['text'] for chunk in all_chunks]
embeddings_list = embeddings.tolist()
metadatas = [{"source": chunk['source'], "chunk_id": chunk['chunk_id']} for chunk in all_chunks]

# Add everything to ChromaDB
collection.add(
    ids=ids,
    documents=texts_list,
    embeddings=embeddings_list,
    metadatas=metadatas
)

print(f"Total chunks added: {collection.count()}")

Total chunks added: 2375


<u>What this does line by line:</u>

**ids** — creates a unique label for each chunk like "property_policy.pdf_42" — combines filename + chunk number so every ID is unique

**texts_list** — pulls just the raw text from all chunks

**embeddings_list** — converts our numpy embeddings to a regular Python list (ChromaDB needs this format)

**metadatas** — stores source filename and chunk number as extra info we can retrieve later

**collection.add(...)** — adds all 4 things for all 2,375 chunks in one shot


Final print confirms how many chunks are now stored

In [14]:
# Peek at the first 3 chunks stored in ChromaDB
result = collection.peek(3)

print("IDs:", result['ids'])
print("\nDocuments:")
for doc in result['documents']:
    print(f"  - {doc[:100]}...")
print("\nMetadata:", result['metadatas'])

IDs: ['commercial_combined_insurance_policy.pdf_0', 'commercial_combined_insurance_policy.pdf_1', 'commercial_combined_insurance_policy.pdf_2']

Documents:
  - FLAT SIZE:  297MM H   420MM W
FINISHED SIZE: 297MM H   210MM W
Commercial  
Combined Insurance
Polic...
  - .............................................................................................3
Navig...
  - ng a Claim - Notification..............................................................................

Metadata: [{'source': 'commercial_combined_insurance_policy.pdf', 'chunk_id': 0}, {'chunk_id': 1, 'source': 'commercial_combined_insurance_policy.pdf'}, {'chunk_id': 2, 'source': 'commercial_combined_insurance_policy.pdf'}]


Now the exciting part — let's ask our first question.

In [15]:
# Ask a question
question = "What is covered under water damage?"

# Convert question to embedding
question_embedding = model.encode(question).tolist()

# Search ChromaDB for most relevant chunks
results = collection.query(
    query_embeddings=[question_embedding],
    n_results=3
)

# Show what came back
print(f"Question: {question}\n")
print("Top 3 most relevant chunks found:\n")
for i in range(3):
    print(f"--- Result {i+1} ---")
    print(f"Source: {results['metadatas'][0][i]['source']}")
    print(f"Chunk ID: {results['metadatas'][0][i]['chunk_id']}")
    print(f"Text: {results['documents'][0][i][:300]}...")
    print()

Question: What is covered under water damage?

Top 3 most relevant chunks found:

--- Result 1 ---
Source: commercial_combined_insurance_policy.pdf
Chunk ID: 73
Text:  organisation.
4 Storm or flood excluding Damage:
A) attributable solely to change in the water table level,
B) caused by frost, subsidence, ground heave or landslip,
C) to fences, gates and moveable property in the open.
5 Escape of water or oil from any tank, apparatus or pipe excluding 
Damage:
A...

--- Result 2 ---
Source: legacy_policy.pdf
Chunk ID: 100
Text: sts and 
expenses in respect of such claim up to £10,000 in total together with 
all costs and expenses incurred with the Company’s written consent
Loss of Metered Water
The Company will pay the addition metered water charges incurred by 
the Insured as a result of Damage caused by any of the Covers...

--- Result 3 ---
Source: office_policy.pdf
Chunk ID: 136
Text: s not acting on behalf of or 
in connection with any political organisation.
3 Storm or flood. 1 

It is purely semantic — not keyword matching.
Here's the proof from your own results:
Your question was: "What is covered under water damage?"
Result 2 from legacy_policy.pdf returned this text:

"Loss of Metered Water... addition metered water charges incurred..."

The words "water damage" don't even appear in that chunk — but it was still returned because the meaning is related to water-related losses in an insurance context.
That's semantic search working.


**How it actually works:**

When you ran model.encode(question), your question became 384 numbers. When we embedded all 2,375 chunks earlier, each chunk also became 384 numbers.
ChromaDB then computes cosine similarity between your question's 384 numbers and every chunk's 384 numbers. The chunks with the highest similarity scores win.

In [16]:
from groq import Groq
from dotenv import load_dotenv
import os

# Load API key from .env file
load_dotenv()

# Initialize Groq client
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

print("Groq client initialized!")
print(f"API key loaded: {os.getenv('GROQ_API_KEY')[:10]}...")

Groq client initialized!
API key loaded: gsk_G79Qaf...


In [17]:
# Build the prompt
def build_prompt(question, retrieved_chunks):
    # Combine the retrieved chunks into one block of context
    context = ""
    for i, chunk in enumerate(retrieved_chunks):
        source = chunk['source']
        text = chunk['text']
        context += f"[Source {i+1}: {source}]\n{text}\n\n"
    
    # Build the full prompt
    prompt = f"""You are an insurance policy assistant. 
Answer the question below using ONLY the context provided.
If the answer is not in the context, say "I could not find this in the provided policy documents."
Always mention which source document your answer comes from.

Context:
{context}

Question: {question}

Answer:"""
    
    return prompt

# Test it
question = "What is covered under water damage?"
retrieved_texts = [
    {"source": results['metadatas'][0][i]['source'],
     "text": results['documents'][0][i]}
    for i in range(3)
]

prompt = build_prompt(question, retrieved_texts)
print(prompt)

You are an insurance policy assistant. 
Answer the question below using ONLY the context provided.
If the answer is not in the context, say "I could not find this in the provided policy documents."
Always mention which source document your answer comes from.

Context:
[Source 1: commercial_combined_insurance_policy.pdf]
 organisation.
4 Storm or flood excluding Damage:
A) attributable solely to change in the water table level,
B) caused by frost, subsidence, ground heave or landslip,
C) to fences, gates and moveable property in the open.
5 Escape of water or oil from any tank, apparatus or pipe excluding 
Damage:
A) by water discharged or leaking from an automatic sprinkler 
installation,
B) in respect of any Building which is empty or not in use.
6 Impact by any road vehicle (including any fork lift truck or o

[Source 2: legacy_policy.pdf]
sts and 
expenses in respect of such claim up to £10,000 in total together with 
all costs and expenses incurred with the Company’s written consen

In [18]:
import httpx
from groq import Groq
from dotenv import load_dotenv
import os

load_dotenv()

# Create Groq client with SSL verification disabled
# This is needed on corporate networks with custom certificates
client = Groq(
    api_key=os.getenv("GROQ_API_KEY"),
    http_client=httpx.Client(verify=False)
)

print("Groq client initialized!")

Groq client initialized!


In [19]:
# Send prompt to Groq LLM
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "user", "content": prompt}
    ],
    temperature=0.1
)

# Extract the answer
answer = response.choices[0].message.content

print(f"Question: {question}")
print(f"\nAnswer:\n{answer}")

Question: What is covered under water damage?

Answer:
I could not find a comprehensive answer to what is covered under water damage in the provided policy documents. However, I found some exclusions related to water damage. 

According to Source 1: commercial_combined_insurance_policy.pdf, exclusions to storm or flood damage include damage attributable solely to change in the water table level, damage caused by frost, subsidence, ground heave or landslip, and damage to fences, gates and moveable property in the open. Additionally, escape of water or oil from any tank, apparatus or pipe is excluded if the damage is caused by water discharged or leaking from an automatic sprinkler installation, or if the building is empty or not in use.

Source 2: legacy_policy.pdf mentions that the Company will pay for additional metered water charges incurred by the Insured as a result of Damage caused by any of the Covers insured under Property Damage Insurance, except for losses not discovered and r

In [20]:
def ask_question(question):
    
    # Step 1 - Embed the question
    #Convert the user's question into 384 numbers so we can compare it against our stored chunk.
    question_embedding = model.encode(question).tolist()
    
    # Step 2 - Retrieve top 3 relevant chunks from ChromaDB
    #Search ChromaDB for the 3 chunks whose fingerprints are closest in meaning to the question's fingerprint. This is the semantic search step.
    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=3
    )
    
    # Step 3 - Build context from retrieved chunks
    #Take the raw results from ChromaDB and organize them into a clean list of dictionaries — each with the chunk text and which document it came from.

    retrieved_chunks = [
        {
            "source": results['metadatas'][0][i]['source'],
            "text": results['documents'][0][i]
        }
        for i in range(3)
    ]
    
    # Step 4 - Build the prompt
    #Combine the 3 chunks + the original question into one structured prompt with our instructions (answer only from context, 
    #cite sources, say I don't know if not found).

    prompt = build_prompt(question, retrieved_chunks)
    
    # Step 5 - Send to Groq and get answer
    #Send the prompt to the Llama model via Groq API and get back a plain English answer.

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0.1
    )
    
    answer = response.choices[0].message.content
    
    # Step 6 - Return answer with sources
    #Collect the answer and source filenames into one neat dictionary and return it — so the caller gets question, answer, 
    #and sources all in one place.

    sources = [chunk['source'] for chunk in retrieved_chunks]
    
    return {
        "question": question,
        "answer": answer,
        "sources": sources
    }

In [21]:
result = ask_question("What is covered under property damage?")
print(result['answer'])
print("\nSources:", result['sources'])

I could not find this in the provided policy documents, as the context only provides information on what is not covered under property damage or general definitions, but does not explicitly state what is covered. (Source 1: commercial_combined_insurance_policy.pdf, Source 2: office_policy.pdf, Source 3: property_owners_policy.pdf)

Sources: ['commercial_combined_insurance_policy.pdf', 'office_policy.pdf', 'property_owners_policy.pdf']


In [29]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create the splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", ".", "!", "?", "", ""]
)

# Test on one document first
test_chunks = text_splitter.split_text(documents["property_policy.pdf"])

print(f"Total chunks created: {len(test_chunks)}")
print(f"\nFirst chunk:")
print(test_chunks[0])
print(f"\nSecond chunk:")
print(test_chunks[1])

Total chunks created: 121

First chunk:
PROPERTY insuRancE
PolicyTHIS POLICY (AND THE SCHEDULE WHICH FORMS AN INTEGRAL PART OF THE POLICY) IS A LEGAL CONTRACT .
IT NEEDS TO BE EXAMINED THOROUGHLY TO ENSURE IT MEETS THE INSURED’S REQUIREMENTS. IF IT DOES NOT
MEET THE INSURED’S REQUIREMENTS THE INSURANCE ADVISER NEEDS TO BE CONTACTED WITHOUT UNDUE
DELAY
ANY FACTS  WHICH  THE INSURER HAS TAKEN INTO ACCOUNT IN THE ASSESSMENT OR  ACCEPTANCE OF THIS
INSURANCE, AND ANY SUBSEQUENT CHANGES TO THOSE FACTS, NEED TO BE DECLARED. FAILURE TO DO SO
MAY INVALIDATE  YOUR POLICY OR RESULT IN CERTAIN COVERS NOT OPERATING FULLY IF YOU ARE IN ANY
DOUBT AS TO WHETHER A FACT IS MATERIAL OR NOT ,  THE INSURANCE ADVISER NEEDS TO BE CONTACTED
WITHOUT UNDUE DELAY
RSA Insurance Ireland DAC (herein called the Company) and the Policyholder agree that
This Policy the Schedule (including any Schedule issued in substitution) and any Memoranda shall be considered one document and any word or

Second chunk:
This Policy 

In [30]:
# Check end of chunk 1 and start of chunk 2
print("End of chunk 1:")
print(test_chunks[0][-100:])
print("\nStart of chunk 2:")
print(test_chunks[1][:100])

End of chunk 1:
 Schedule issued in substitution) and any Memoranda shall be considered one document and any word or

Start of chunk 2:
This Policy the Schedule (including any Schedule issued in substitution) and any Memoranda shall be 


In [31]:
import re

def clean_text(text):
    # Fix multiple spaces
    text = re.sub(r' +', ' ', text)
    # Fix multiple newlines — normalize to double newline
    text = re.sub(r'\n{3,}', '\n\n', text)
    # Add period after ALL CAPS headings so splitter sees them as sentence ends
    text = re.sub(r'([A-Z]{5,})\n', r'\1.\n\n', text)
    # Strip leading/trailing whitespace
    text = text.strip()
    return text

# Test on property policy
cleaned = clean_text(documents["property_policy.pdf"])

# Now chunk the cleaned text
test_chunks = text_splitter.split_text(cleaned)

print(f"Total chunks: {len(test_chunks)}")
print(f"\nFirst chunk:")
print(test_chunks[0])
print(f"\nSecond chunk:")
print(test_chunks[1])

Total chunks: 120

First chunk:
PROPERTY insuRancE
PolicyTHIS POLICY (AND THE SCHEDULE WHICH FORMS AN INTEGRAL PART OF THE POLICY) IS A LEGAL CONTRACT .
IT NEEDS TO BE EXAMINED THOROUGHLY TO ENSURE IT MEETS THE INSURED’S REQUIREMENTS. IF IT DOES NOT
MEET THE INSURED’S REQUIREMENTS THE INSURANCE ADVISER NEEDS TO BE CONTACTED WITHOUT UNDUE.

DELAY.

ANY FACTS WHICH THE INSURER HAS TAKEN INTO ACCOUNT IN THE ASSESSMENT OR ACCEPTANCE OF THIS
INSURANCE, AND ANY SUBSEQUENT CHANGES TO THOSE FACTS, NEED TO BE DECLARED. FAILURE TO DO SO
MAY INVALIDATE YOUR POLICY OR RESULT IN CERTAIN COVERS NOT OPERATING FULLY IF YOU ARE IN ANY
DOUBT AS TO WHETHER A FACT IS MATERIAL OR NOT , THE INSURANCE ADVISER NEEDS TO BE CONTACTED.

WITHOUT UNDUE DELAY.

Second chunk:
RSA Insurance Ireland DAC (herein called the Company) and the Policyholder agree that
This Policy the Schedule (including any Schedule issued in substitution) and any Memoranda shall be considered one document and any word or
expression to whic

In [32]:
# Check end of chunk 1 and start of chunk 2
print("End of chunk 1:")
print(test_chunks[0][-100:])
print("\nStart of chunk 2:")
print(test_chunks[1][:100])

End of chunk 1:
ETHER A FACT IS MATERIAL OR NOT , THE INSURANCE ADVISER NEEDS TO BE CONTACTED.

WITHOUT UNDUE DELAY.

Start of chunk 2:
RSA Insurance Ireland DAC (herein called the Company) and the Policyholder agree that
This Policy th


In [33]:
from sentence_transformers import SentenceTransformer
import chromadb

# Rechunk all documents with cleaned text
all_chunks_new = []

for filename, text in documents.items():
    cleaned = clean_text(text)
    chunks = text_splitter.split_text(cleaned)
    
    for i, chunk in enumerate(chunks):
        all_chunks_new.append({
            "text": chunk,
            "source": filename,
            "chunk_id": i
        })
    print(f"{filename}: {len(chunks)} chunks")

print(f"\nOld total: 2375 chunks")
print(f"New total: {len(all_chunks_new)} chunks")

commercial_combined_insurance_policy.pdf: 394 chunks
freight_solutions_policy.pdf: 73 chunks
legacy_policy.pdf: 254 chunks
office_policy.pdf: 303 chunks
property_owners_policy.pdf: 129 chunks
property_policy.pdf: 120 chunks

Old total: 2375 chunks
New total: 1273 chunks


In [34]:
# Delete old collection
client = chromadb.PersistentClient(path="../data/chromadb")
client.delete_collection("insurance_policies")

# Create fresh collection
collection = client.get_or_create_collection(
    name="insurance_policies",
    metadata={"hnsw:space": "cosine"}
)

print("Fresh collection created!")
print(f"Count: {collection.count()}")

Fresh collection created!
Count: 0
